<a href="https://colab.research.google.com/github/dynamodenis/QDrant/blob/main/hnsw_indexing/index_and_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HNSW Indexing, combining Vector Search and Filtering

## The challenge
Consider retrieving items from an online store collection where you only want to show laptops priced under $1,000. That price information, along with the category ’laptop’, isn’t part of the vector - it lives in the payload.

When you apply a filter like price < 1000, you’re essentially restricting which points are eligible during search. This introduces challenges for graph traversal because HNSW depends on both short- and long-range edges to efficiently explore the vector space. It requires any point in the graph to be reachable. But if filtering removes a large portion of those points, the search path can break. You risk missing relevant results - not because they weren’t similar, but because they were unreachable under the filter.

## Qdrant's Solution: Filterable HNSW
Qdrant solves this with a smarter approach. We guarantee that the HNSW graph remains connected by creating additional edges to maintain connectivity under filtering. Qdrant builds subgraphs per payload value, then merges them back into the full graph.

So if you filter brand = Apple, Qdrant has already built a connected subgraph of just Apple points, and traversal works fine within that subset.

In [ ]:
!pip install qdrant-client

In [19]:
from qdrant_client import QdrantClient, models
from google.colab import userdata
import os

client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))
# For Colab:
# from google.colab import userdata
# client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

collection_name = "store"
vector_size = 768

if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE,
    ),
    optimizers_config=models.OptimizersConfigDiff(
        indexing_threshold=100,
    ),
)

# Index frequently filtered fields
client.create_payload_index(
    collection_name=collection_name,
    field_name="category",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="price",
    field_schema=models.PayloadSchemaType.FLOAT,
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="brand",
    field_schema=models.PayloadSchemaType.KEYWORD,
)


UpdateResult(operation_id=6, status=<UpdateStatus.COMPLETED: 'completed'>)

In [20]:
# Upload data
import random

points = []
for i in range(1000):
    points.append(
        models.PointStruct(
            id=i,
            vector=[random.random() for _ in range(vector_size)],
            payload={
                "category": random.choice(["laptop", "phone", "tablet"]),
                "price": random.randint(0, 1000),
                "brand": random.choice(
                    ["Apple", "Dell", "HP", "Lenovo", "Asus", "Acer", "Samsung"]
                ),
            },
        )
    )
client.upload_points(
    collection_name=collection_name,
    points=points,
)


## Memory Considerations

Payload indexes consume additional memory, so it’s recommended to only index fields used in filtering conditions.

If memory is limited, prioritise the field that produces the most specific search results. The more varied and granular the payload field values are, the more effective it’s index will be.

In [22]:
# Setting Up Filterable Search
# Create filter combining multiple conditions
filter_conditions = models.Filter(
    must=[
        models.FieldCondition(key="category", match=models.MatchValue(value="laptop")),
        models.FieldCondition(key="price", range=models.Range(lte=1000)),
        models.FieldCondition(key="brand", match=models.MatchAny(any=["Apple", "Dell", "HP"])),
    ]
)

query_vector = [random.random() for _ in range(vector_size)]

# Execute filtered search
results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    query_filter=filter_conditions,
    limit=10,
    search_params=models.SearchParams(hnsw_ef=128),
)

print(results)


points=[ScoredPoint(id=295, version=11, score=0.7712363, payload={'category': 'laptop', 'price': 258, 'brand': 'Dell'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=890, version=20, score=0.7700931, payload={'category': 'laptop', 'price': 950, 'brand': 'HP'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=824, version=19, score=0.76733744, payload={'category': 'laptop', 'price': 436, 'brand': 'Dell'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=372, version=12, score=0.7656456, payload={'category': 'laptop', 'price': 905, 'brand': 'Apple'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=923, version=21, score=0.7656069, payload={'category': 'laptop', 'price': 865, 'brand': 'Apple'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=498, version=14, score=0.76452047, payload={'category': 'laptop', 'price': 965, 'brand': 'HP'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=6, version=7, sco